# Session 04: Edge Detection & Feature Detection

**CVI4IC — Summer Semester 2026**

Topics: Sobel gradients, Canny edge detection, Hough transform, Harris corners, SIFT keypoints, and feature matching.

In [ ]:
!pip install -q opencv-python-headless matplotlib numpy

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

In [ ]:
!git clone --depth 1 https://github.com/fruits-360/fruits-360-100x100.git 2>/dev/null || echo "Dataset already cloned."

DATASET_ROOT = Path("fruits-360-100x100")
TRAIN_DIR = DATASET_ROOT / "Training"

classes = sorted(os.listdir(TRAIN_DIR))
print(f"Fruits-360: {len(classes)} classes")

def load_fruit(class_name, index=0):
    cls_dir = TRAIN_DIR / class_name
    return cv2.imread(str(sorted(cls_dir.glob("*.jpg"))[index]))

def to_rgb(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

def to_gray(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

def show_image(ax, image, title, cmap=None):
    ax.imshow(image, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")

## 1. Gradient-Based Edge Detection

Edge detection starts with image derivatives. We use Sobel filters to estimate horizontal and vertical gradients.

In [ ]:
img = load_fruit("Apple Red 1")
gray = to_gray(img)

sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
grad_mag = np.sqrt(sobel_x**2 + sobel_y**2)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
show_image(axes[0], to_rgb(img), "Original")
show_image(axes[1], np.abs(sobel_x), "|Sobel X|", cmap="gray")
show_image(axes[2], np.abs(sobel_y), "|Sobel Y|", cmap="gray")
show_image(axes[3], grad_mag, "Gradient Magnitude", cmap="gray")
plt.tight_layout()
plt.show()

## 2. Canny Edge Detection

Canny combines Gaussian smoothing, gradient estimation, non-maximum suppression, and hysteresis thresholding.

In [ ]:
img = load_fruit("Tomato 1")
gray = to_gray(img)

blurred = cv2.GaussianBlur(gray, (5, 5), 1.2)
edge_lo = cv2.Canny(blurred, 40, 100)
edge_mid = cv2.Canny(blurred, 80, 160)
edge_hi = cv2.Canny(blurred, 120, 220)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
show_image(axes[0], gray, "Grayscale", cmap="gray")
show_image(axes[1], edge_lo, "Canny (40, 100)", cmap="gray")
show_image(axes[2], edge_mid, "Canny (80, 160)", cmap="gray")
show_image(axes[3], edge_hi, "Canny (120, 220)", cmap="gray")
plt.tight_layout()
plt.show()

## 3. Hough Transform for Line Detection

The Hough transform maps edge pixels from image space into a parameter space and finds lines through voting. We use a Canny edge map as input and then detect line segments with `cv2.HoughLinesP`.

In [ ]:
img = cv2.imread("foto1A.jpg")
if img is None:
    os.system('curl -L --fail --silent --show-error -o "foto1A.jpg" "https://www.ic.unicamp.br/~helio/imagens_registro/foto1A.jpg"')
    img = cv2.imread("foto1A.jpg")

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(gray, 80, 180)

lines = cv2.HoughLinesP(
    edges,
    rho=1,
    theta=np.pi / 180,
    threshold=80,
    minLineLength=60,
    maxLineGap=10,
)

line_overlay = img.copy()
if lines is not None:
    for line in lines:
        x1, y1, x2, y2 = line[0]
        cv2.line(line_overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
show_image(axes[0], gray, "Grayscale input", cmap="gray")
show_image(axes[1], edges, "Canny edges", cmap="gray")
show_image(axes[2], to_rgb(line_overlay), "Hough line segments")
plt.tight_layout()
plt.show()

print("Detected line segments:", 0 if lines is None else len(lines))

## 4. Harris Corner Detection

Corners are points with strong intensity changes in two directions. Harris uses the local second-moment matrix to score corner-like structures.

In [ ]:
img = load_fruit("Strawberry 1")
gray = np.float32(to_gray(img))

harris = cv2.cornerHarris(gray, blockSize=5, ksize=3, k=0.04)
harris = cv2.dilate(harris, None)

overlay = img.copy()
overlay[harris > 0.01 * harris.max()] = [0, 0, 255]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
show_image(axes[0], to_gray(img), "Input", cmap="gray")
show_image(axes[1], np.log1p(np.maximum(harris, 0)), "Harris response (log)", cmap="hot")
show_image(axes[2], to_rgb(overlay), "Corners overlaid")
plt.tight_layout()
plt.show()

## 5. SIFT Keypoints and Descriptors

SIFT detects scale-invariant keypoints and computes 128-dimensional descriptors that can be matched across images.

In [ ]:
img = load_fruit("Kiwi 1")
gray = to_gray(img)

sift = cv2.SIFT_create()
kps_sift, des_sift = sift.detectAndCompute(gray, None)
vis_sift = cv2.drawKeypoints(gray, kps_sift, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
show_image(axes[0], gray, "Input", cmap="gray")
show_image(axes[1], vis_sift, f"SIFT keypoints: {len(kps_sift)}", cmap="gray")
plt.tight_layout()
plt.show()

print("SIFT descriptor shape:", des_sift.shape if des_sift is not None else None)

## 6. Feature Matching Between Two Views

We detect SIFT keypoints in two images, match descriptors with a brute-force matcher, and filter matches using Lowe's ratio test.

In [ ]:
# 6) Feature matching between two views
!curl -L --fail --silent --show-error -o "foto1A.jpg" "https://www.ic.unicamp.br/~helio/imagens_registro/foto1A.jpg"
!curl -L --fail --silent --show-error -o "foto1B.jpg" "https://www.ic.unicamp.br/~helio/imagens_registro/foto1B.jpg"

img_a = cv2.imread("foto1A.jpg")
img_b = cv2.imread("foto1B.jpg")
gray_a = cv2.cvtColor(img_a, cv2.COLOR_BGR2GRAY)
gray_b = cv2.cvtColor(img_b, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
kps_a, des_a = sift.detectAndCompute(gray_a, None)
kps_b, des_b = sift.detectAndCompute(gray_b, None)

bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
raw_matches = bf.knnMatch(des_a, des_b, k=2)

ratio = 0.75
good_matches = [m for m, n in raw_matches if m.distance < ratio * n.distance]
good_matches = sorted(good_matches, key=lambda x: x.distance)

print(f"SIFT keypoints A: {len(kps_a)}")
print(f"SIFT keypoints B: {len(kps_b)}")
print(f"Good matches after ratio test: {len(good_matches)}")

match_vis = cv2.drawMatches(
    img_a,
    kps_a,
    img_b,
    kps_b,
    good_matches[:80],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)

plt.figure(figsize=(18, 8))
plt.imshow(cv2.cvtColor(match_vis, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("SIFT + BFMatcher + Lowe ratio test")
plt.tight_layout()
plt.show()

In [ ]:
distances = np.array([m.distance for m in good_matches], dtype=np.float32)

plt.figure(figsize=(8, 4))
plt.hist(distances, bins=30)
plt.xlabel("SIFT match distance")
plt.ylabel("Count")
plt.title("Distribution of good SIFT match distances")
plt.tight_layout()
plt.show()

if len(distances) > 0:
    print("Distance stats -> min:", float(distances.min()), "median:", float(np.median(distances)), "max:", float(distances.max()))

## 7. Geometric Verification with RANSAC

Feature matches contain outliers. A robust homography estimation with RANSAC filters incorrect correspondences.

In [ ]:
if len(good_matches) >= 4:
    pts_a = np.float32([kps_a[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    pts_b = np.float32([kps_b[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    H, inlier_mask = cv2.findHomography(pts_a, pts_b, cv2.RANSAC, 3.0)
    inliers = inlier_mask.ravel().astype(bool)

    print(f"Inliers: {inliers.sum()} / {len(inliers)}")

    inlier_matches = [m for m, keep in zip(good_matches, inliers) if keep]
    inlier_vis = cv2.drawMatches(
        img_a,
        kps_a,
        img_b,
        kps_b,
        inlier_matches[:80],
        None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    )

    plt.figure(figsize=(18, 8))
    plt.imshow(cv2.cvtColor(inlier_vis, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("RANSAC inlier matches")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough matches for homography estimation.")

## ✏️ Exercises

### Exercise 1: Compare Sobel, Scharr, and Canny

1. Load one Fruits-360 image and convert it to grayscale.
2. Compute edge maps using Sobel, Scharr, and Canny.
3. Plot all results side by side.
4. Discuss where each method works best and where it fails.

**Hint:** try changing Canny thresholds and compare stability.

### Exercise 2: Harris vs SIFT Keypoints

1. Select two different fruit classes with visible texture.
2. Detect corners with Harris and keypoints with SIFT.
3. Visualize all detections on top of the images.
4. Compare detector behavior for scale changes and blur.

**Goal:** Explain the difference between a corner detector and a keypoint+descriptor pipeline.

### Exercise 3: Robust SIFT Matching Mini-Challenge

1. Use `foto1A.jpg` and `foto1B.jpg` (or your own pair of overlapping images).
2. Compute SIFT matches with different Lowe ratio values (e.g., 0.6, 0.7, 0.8).
3. Which ratio setting is best for this image pair, and why?

In [ ]:
# Your code here